# Import Packages

In [1]:
# packages

import argparse
import os
import sys
from pathlib import Path

from PIL import Image as PILImage
from reportlab.lib.pagesizes import landscape, letter
from reportlab.lib import colors
from reportlab.pdfgen import canvas

In [7]:
# input variables

input_dir = 'output/2025_november/'

In [ ]:
# Constants

CARD_W_IN = 4.0
CARD_H_IN = 6.0
MARGIN_IN = 0.25
GUTTER_IN = 0.25
TICK_LEN_IN = 0.125
TICK_OFFSET_IN = 0.0625
DPI = 72

In [10]:
def inches(val: float) -> float:
    return val * DPI

In [11]:
def find_card_pairs(
    input_dir: str
) -> list[tuple[str, Path, Path]]:
    card_path = Path(input_dir)
    healthy = {f.stem.replace('_healthy',''): 
               f for f in card_path.glob('*_healthy.jpg')}
    injured = {f.stem.replace('_injured',''):
               f for f in card_path.glob('*_injured.jpg')}
    all_names = sorted(set(healthy) | set(injured))
    pairs = []
    for name in all_names:
        if name in healthy and name in injured:
            pairs.append((name, healthy[name], injured[name]))
        elif name in healthy:
            print(f'   Warning: No injured file for "{name}" - skipping.')
        elif name in injured:
            print(f'   Warning: No healthy file for "{name}" - skipping.')

    return pairs

In [12]:
def draw_cut_marks(
    c: canvas.Canvas,
    x: float,
    y: float,
    w: float,
    h: float,
    tick: float | None = None,
    offset: float | None = None,
) -> None:
    
    if tick is None:
        tick = inches(TICK_LEN_IN)
    if offset is None:
        offset = inches(TICK_OFFSET_IN)

    c.setStrokeColor(colors.black)
    c.setLineWidth(0.5)

    corners = [
        (x , y),        # bottom-left
        (x + w, y),     # bottom-right
        (x, y + h),     # top-left
        (x + w, y + h)  # top-right
    ]

    for cx, cy in corners:
        # determine tick directions (inward from corner)
        dx = 1 if cx == x else -1
        dy = 1 if cy == y else -1

        # horizontal tick
        hx0 = cx + dx * offset
        hx1 = cx + dx * (offset + tick)
        c.line(h0, cy, hx1, cy)

        # vertical tick
        vy0 = cy + dy * offset
        vy1 = cy + dy * offset
        c.line(cx, vy0, cx, vy1)

    # return nothing

In [ ]:
def place_cards(
    c: canvas.Canvas,
    image_path: Path,
    x: float,
    y: float,
    w: float,
    h: float,
    flip_horizontal: bool | False = False
)
    img = PILImage.open(image_path).convert('RGB')

    if flip_horizontal:
        img = img.transpose(PILImage.FLIP_LEFT_RIGHT)

    # Scale to fit
    img_w, img_h = img.size
    scale = min(w / img_w, h / img_h)
    new_w = img_w * scale
    new_h = img_h * scale

    # Center within card slot
    